In [15]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
# Load the dataset
df = pd.read_csv('/content/features_rahim.csv')

In [17]:
df.isna().sum()

,0
product_name,1
discounted_price,3634
actual_price,3634
rating,0
rating_count,1658
Category,0
product_link,0
product_name_missing,0
discount_percentage,0
avg_price_category,0


In [22]:
df = df.dropna(subset=["product_name"])

In [24]:
df = df.dropna(subset=["actual_price", "discounted_price"], how="all")

In [26]:
#Fill missing rating_count with the mean value
df["rating_count"].fillna(df["rating_count"].mean(), inplace=True)

<ipython-input-26-c66f34c61869>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["rating_count"].fillna(df["rating_count"].mean(), inplace=True)


In [27]:
df.isna().sum()

,0
product_name,0
discounted_price,0
actual_price,0
rating,0
rating_count,0
Category,0
product_link,0
product_name_missing,0
discount_percentage,0
avg_price_category,0


In [28]:
df.head()

,product_name,discounted_price,actual_price,rating,rating_count,Category,product_link,product_name_missing,discount_percentage,avg_price_category,price_difference_category,is_high_discount,cleaned_product_name
0,Wayona Nylon Braided USB to Lightning Fast Cha...,399.0,1099.0,4.2,24269.0,Computers&Accessories|Accessories&Peripherals|...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,False,63.694268,360.723691,38.276309,1,wayona nylon braided usb to lightning fast cha...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,199.0,349.0,4,43994.0,Computers&Accessories|Accessories&Peripherals|...,https://www.amazon.in/Ambrane-Unbreakable-Char...,False,42.979943,360.723691,-161.723691,0,ambrane unbreakable 60w 3a fast charging 1 5...
2,Sounce Fast Phone Charging Cable & Data Sync U...,199.0,1899.0,3.9,7928.0,Computers&Accessories|Accessories&Peripherals|...,https://www.amazon.in/Sounce-iPhone-Charging-C...,False,89.520800,360.723691,-161.723691,1,sounce fast phone charging cable data sync u...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,329.0,699.0,4.2,94363.0,Computers&Accessories|Accessories&Peripherals|...,https://www.amazon.in/Deuce-300-Resistant-Tang...,False,52.932761,360.723691,-31.723691,1,boat deuce usb 300 2 in 1 type c micro usb s...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,154.0,399.0,4.2,16905.0,Computers&Accessories|Accessories&Peripherals|...,https://www.amazon.in/Portronics-Konnect-POR-1...,False,61.403509,360.723691,-206.723691,1,portronics konnect l 1 2m fast charging 3a 8 p...


In [29]:
# Extract brand names (assuming the first word is the brand)
df["brand_name"] = df["cleaned_product_name"].apply(lambda x: x.split()[0] if isinstance(x, str) else "")

In [30]:
# Convert product names to numerical vectors using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["cleaned_product_name"])

In [32]:
# Compute cosine similarity between all products
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [42]:
# Function to recommend products
def recommend_products(user_input, top_n=5):
    user_input = user_input.lower()

    # Check if input matches a known brand
    if user_input in df["brand_name"].str.lower().values:
        return df[df["brand_name"].str.lower() == user_input][["cleaned_product_name", "discounted_price", "actual_price"]].head(top_n)

    # Check if input is a product type (search for similar words)
    matching_products = df[df["cleaned_product_name"].str.contains(user_input, case=False, na=False)]
    if not matching_products.empty:
        return matching_products[["cleaned_product_name", "discounted_price", "actual_price"]].head(top_n)

    # Otherwise, find the most similar product
    if user_input not in df["cleaned_product_name"].str.lower().values:
        return "Product not found in dataset."

    product_idx = df[df["cleaned_product_name"].str.lower() == user_input].index[0]
    sim_scores = list(enumerate(cosine_sim[product_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1 : top_n + 1]

    recommended_indices = [i[0] for i in sim_scores]
    return df.iloc[recommended_indices][["cleaned_product_name", "discounted_price", "actual_price"]]

In [47]:
print(recommend_products("vaccum"))

                                   cleaned_product_name  discounted_price  \
1323  inalsa vaccum cleaner handheld 800w high power...            1799.0   

      actual_price  
1323        3295.0  
